# Information coefficients (Prompt #8, Steps 14-30)

Rank IC of every feature with every stored target, with Newey-West standard errors from monthly sufficient statistics, BH q-values per family, and the conditions each status reads: the studentized max-T circular-shift null, the pipeline nulls (a residual target needs the random-walk control - invariant 9) and yearly sign consistency.

**This notebook contains no calculations.** Everything is computed by `xauusd_quant.features` / `xauusd_quant.alpha` / `xauusd_quant.research.feature_*` and read back from `results/feature_research/`. Generate those first:

```
xq build-feature-factory --timeframe 1h      # versioned causal matrix + targets
xq feature-research --timeframe 1h           # every stage, statuses, ledger, figures
xq feature-research --all                    # 5m, 15m, 30m, 1h (one child process each)
xq feature-research --compare                # cross-timeframe tables
```

**Descriptive research only.** Statuses (`strong_candidate` ... `failed_null`) are statistical information about rank ICs against circular-shift, pipeline and noise-feature nulls - not signals, not a model, no cost, no PnL. Features at `t` use bars `<= t`; targets live in their own namespace (`target_*`) and are never joined into the matrix.

In [ ]:
import sys
import json
from pathlib import Path

# Work against the repository's own source tree, not an installed copy.
ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import polars as pl
from IPython.display import Image, display

from xauusd_quant.features.factory import current_version_dir
from xauusd_quant.features.factory_config import load_features_config

fcfg = load_features_config(ROOT / "config" / "features.yaml")

TIMEFRAME = "1h"       # 5m | 15m | 30m | 1h - change and re-run

base = fcfg.results_path / TIMEFRAME
if not base.exists():
    print(f"No research for {TIMEFRAME}. Run:  xq feature-research --timeframe {TIMEFRAME}")

def table(name):
    """Load a previously generated research table (None if it was not written)."""
    path = base / f"{name}.parquet"
    return pl.read_parquet(path) if path.exists() else None

def plot(name):
    path = base / "figures" / f"{name}.png"
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f"(no figure {path.name})")

def load_json(name):
    path = base / name
    return json.loads(path.read_text()) if path.exists() else {}

summary = load_json("summary.json")
{k: v for k, v in summary.items() if not isinstance(v, (dict, list))}

## Status counts

In [ ]:
summary.get('status_counts')

## Evidence per feature x target (pooled rank IC)

In [ ]:
ev = table('feature_evidence')
ev.select('feature', 'target', 'value', 'se', 'q_value', 'beyond_shift_null', 'beyond_pipeline_null',
          'mechanical_share', 'sign_consistency', 'recent_ic', 'verdict').sort(pl.col('value').abs(), descending=True).head(30)

## Statuses

In [ ]:
table('feature_status').select('feature', 'family', 'status', 'best_kind', 'best_target', 'best_abs_rank_ic', 'reason')

## Yearly and rolling IC of the leading candidates

In [ ]:
plot('yearly_ic')
plot('rolling_ic')

## Conditioning: volatility terciles, regimes, intraday

In [ ]:
display(table('conditioning/volatility_conditioning'))
display(table('conditioning/regime_conditioning'))
table('conditioning/intraday_ic')

## Deciles (full-sample edges: descriptive only)

In [ ]:
plot('deciles')
table('deciles/monotonicity')

## Mutual information (copula, Miller-Madow)

In [ ]:
table('mi/mutual_information')